# Concatenate Indoor Walk Test CSV Files

This notebook combines every `.CSV` file in the `CSV` folder into `Concat_Indoor_Walk_Test_from_csv.csv` without dropping any measurement rows.

In [ ]:
from __future__ import annotations

import csv
from itertools import islice
from pathlib import Path

ROOT = Path.cwd()
CSV_DIR = ROOT / 'CSV'
OUTPUT_FILE = ROOT / 'Concat_Indoor_Walk_Test_from_csv.csv'

if not CSV_DIR.exists():
    raise FileNotFoundError(f'Missing input folder: {CSV_DIR}')

csv_files = sorted(CSV_DIR.glob('*.CSV'))
if not csv_files:
    raise FileNotFoundError('No .CSV files found in CSV/')

print(f'csv_files={len(csv_files)}')
print(f'output_file={OUTPUT_FILE}')

In [ ]:
fieldnames: list[str] = []
fieldname_set: set[str] = set()
total_rows = 0

for csv_file in csv_files:
    with csv_file.open('r', encoding='utf-8', errors='replace', newline='') as source:
        reader = csv.DictReader(source)
        if reader.fieldnames is None:
            continue

        for fieldname in reader.fieldnames:
            if fieldname not in fieldname_set:
                fieldname_set.add(fieldname)
                fieldnames.append(fieldname)

        for _ in reader:
            total_rows += 1

with OUTPUT_FILE.open('w', encoding='utf-8', newline='') as destination:
    writer = csv.DictWriter(destination, fieldnames=fieldnames, extrasaction='ignore')
    writer.writeheader()

    written_rows = 0
    for csv_file in csv_files:
        with csv_file.open('r', encoding='utf-8', errors='replace', newline='') as source:
            reader = csv.DictReader(source)
            if reader.fieldnames is None:
                continue

            for row in reader:
                writer.writerow({fieldname: row.get(fieldname, '') for fieldname in fieldnames})
                written_rows += 1

print(f'columns={len(fieldnames)}')
print(f'input_rows={total_rows}')
print(f'output_rows={written_rows}')

if written_rows != total_rows:
    raise RuntimeError('Output row count does not match input row count')

In [ ]:
with OUTPUT_FILE.open('r', encoding='utf-8', errors='replace', newline='') as merged_file:
    reader = csv.DictReader(merged_file)
    preview_rows = list(islice(reader, 5))

print(f'preview_rows={len(preview_rows)}')
for index, row in enumerate(preview_rows, start=1):
    print(f'Row {index}:')
    print({key: value for key, value in row.items() if value})
    print()